###Prof. Fernando Amaral https://www.eia.ai/

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import Tokenizer, StringIndexer, Word2Vec
spark = SparkSession.builder.appName("nlp").getOrCreate()

In [ ]:
spam = spark.sql("select * from spam")

In [ ]:
spam.show(5)

In [ ]:
stringmodel = StringIndexer(inputCol="Category",outputCol="CategoryIndex")
spamnew = stringmodel.fit(spam).transform(spam)
spamnew.show(5)

In [ ]:
tokens = Tokenizer(inputCol="Message", outputCol="MessageToken")
spamtoken = tokens.transform(spamnew)

In [ ]:
spamtoken.select("MessageToken").show(5)

In [ ]:
word2vec = Word2Vec(inputCol="MessageToken", outputCol="Messagew2v")
spamresult = word2vec.fit(spamtoken).transform(spamtoken)
spamresult.show(5)

In [ ]:
spamresult.select("Messagew2v").show(5)

In [ ]:
spamTreino,spamTeste = spamresult.randomSplit([0.7,0.3])

In [ ]:
rf = RandomForestClassifier(labelCol="CategoryIndex", featuresCol="Messagew2v", numTrees=500)
modelo = rf.fit(spamTreino)

In [ ]:
previsoes = modelo.transform(spamTeste)

In [ ]:
previsoes.show(10)

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
avaliar = BinaryClassificationEvaluator(rawPredictionCol="prediction", labelCol="CategoryIndex", metricName="areaUnderROC")
areaUnderRoc = avaliar.evaluate(previsoes)
print(areaUnderRoc)